# 05 — Model Training: PCOS Symptoms Dataset

**Goal:** Train multiple regressors to predict **PCOS Risk Score** (continuous 0–100 %) from symptom data, then derive Low / Medium / High labels at inference time.

| Step | What | Why |
|------|------|-----|
| 1 | Load processed data | Already encoded + SMOTE-balanced from FE notebook |
| 2 | Build continuous risk score target | Map binary PCOS + symptom burden → float [0, 1] |
| 3 | Train 3 regressors | Ridge, Random Forest, XGBoost |
| 4 | Evaluate with regression metrics | MAE, RMSE, R² |
| 5 | Cross-validate best models | Guard against lucky train/test split |
| 6 | Save all trained models | Evaluation notebook loads these for comparison |

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

# ── Sklearn ────────────────────────────────────────────────
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.model_selection import cross_val_score, KFold
from xgboost import XGBRegressor

os.makedirs("../models/symptoms", exist_ok=True)
print("Libraries loaded ✓")

Libraries loaded ✓


## 1. Load Processed Data

In [2]:
X_train = pd.read_csv("../data/processed/symptoms_X_train.csv")
y_train_raw = pd.read_csv("../data/processed/symptoms_y_train.csv").squeeze()
X_test  = pd.read_csv("../data/processed/symptoms_X_test.csv")
y_test_raw  = pd.read_csv("../data/processed/symptoms_y_test.csv").squeeze()

print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"Features      : {list(X_train.columns)}")

X_train shape : (830, 30)
X_test shape  : (137, 30)
Features      : ['Family_background', 'LH_hormone', 'FSH_hormone', 'Diabetes_measurment', 'TSH_hormone', 'Prolactin_hormone', 'Hemoglobin_level', 'Cyst_ovary', 'Diagnosis_age', 'Overweight', 'Period_type', 'Hormonal_imbalance', 'Gain_weight', 'Excess_facial_hair', 'Excess_body_hair', 'Dark_area', 'Pimple_face', 'Hormonal_acne_face', 'Blood_pressure', 'Fast_food', 'Losing_hair', 'Mood_swing_period', 'Craving_PT', 'Depress', 'Mental_stress', 'Insomnia', 'Slow_activity', 'Weight(kg)', 'Height(m)', 'BMI(kg/m*m)']


## 2. Build Continuous Risk Score Target

Instead of 3 discrete classes, we produce a **float score in [0, 1]** per patient (displayed as 0 – 100 %).

| PCOS label | Score range | Zone |
|-----------|-------------|------|
| 0 (negative) | [0.00, 0.33) | Low |
| 1 (positive), low burden | [0.33, 0.66) | Medium |
| 1 (positive), high burden | [0.66, 1.00] | High |

**Burden** = fraction of high-signal features that are active (> 0).  
High-signal features (from Cramér's V EDA): `Period_type`, `Excess_facial_hair`, `Excess_body_hair`, `Dark_area`, `Hormonal_imbalance`, `Cyst_ovary`, `Overweight`, `Gain_weight`, `Insomnia`, `Losing_hair`.

In [3]:
# High-signal symptom columns (confirmed by Cramér's V in EDA)
HIGH_SIGNAL_COLS = [
    col for col in [
        "Period_type",
        "Excess_facial_hair",
        "Excess_body_hair",
        "Dark_area",
        "Hormonal_imbalance",
        "Cyst_ovary",
        "Overweight",
        "Gain_weight",
        "Insomnia",
        "Losing_hair",
    ] if col in X_train.columns
]
print(f"High-signal columns found: {HIGH_SIGNAL_COLS}")


def assign_risk_score(X, y_binary):
    """
    Returns a continuous float score in [0, 1] per patient.

    - PCOS = 0  →  score ∈ [0.00, 0.33)   Low zone
                   (even negatives vary slightly by how many
                    incidental signals they carry)
    - PCOS = 1  →  score ∈ [0.33, 1.00]
        burden_norm = 0  →  0.33  (just crossed the positive threshold)
        burden_norm = 1  →  1.00  (all high-signal features active)
    """
    y_binary    = np.array(y_binary)
    n_signals   = len(HIGH_SIGNAL_COLS)

    # Normalised burden: fraction of high-signal features that are active (>0)
    burden_norm = X[HIGH_SIGNAL_COLS].gt(0).sum(axis=1).values / n_signals

    score = np.where(
        y_binary == 0,
        burden_norm * 0.33,          # Negative: slight variation, stays in Low zone
        0.33 + burden_norm * 0.67    # Positive: maps burden linearly into [0.33, 1.00]
    )
    return score.clip(0.0, 1.0)


y_train = assign_risk_score(X_train, y_train_raw)
y_test  = assign_risk_score(X_test,  y_test_raw)

# Quick sanity check — show score distribution
def score_dist(y, label):
    low    = (y <  0.33).sum()
    medium = ((y >= 0.33) & (y < 0.66)).sum()
    high   = (y >= 0.66).sum()
    print(f"  {label} → Low: {low}  Medium: {medium}  High: {high}  "
          f"(mean={y.mean():.3f}, std={y.std():.3f})")

print("Risk Score Distribution (by zone):")
score_dist(y_train, "TRAIN")
score_dist(y_test,  "TEST ")

High-signal columns found: ['Period_type', 'Excess_facial_hair', 'Excess_body_hair', 'Dark_area', 'Hormonal_imbalance', 'Cyst_ovary', 'Overweight', 'Gain_weight', 'Insomnia', 'Losing_hair']
Risk Score Distribution (by zone):
  TRAIN → Low: 415  Medium: 22  High: 393  (mean=0.469, std=0.391)
  TEST  → Low: 33  Medium: 8  High: 96  (mean=0.681, std=0.340)


## 3. Define Regressors

| Model | Why include it |
|-------|----------------|
| Ridge Regression | Fast linear baseline; interpretable coefficients |
| Random Forest | Handles non-linearity; built-in feature importance |
| XGBoost | Best gradient boosting; usually top performer |

In [4]:
models = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ),
    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        random_state=42,
        n_jobs=-1,
    ),
}
print(f"{len(models)} regressors defined ✓")

3 regressors defined ✓


## 4. Train & Evaluate All Models

In [6]:
results = {}

for name, model in models.items():
    print(f"\n{'─'*55}")
    print(f"  Training: {name}")
    print(f"{'─'*55}")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test).clip(0.0, 1.0)   # keep scores in valid range

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    results[name] = {
        "model":  model,
        "MAE":    round(mae,  4),
        "RMSE":   round(rmse, 4),
        "R2":     round(r2,   4),
        "y_pred": y_pred,
    }

    print(f"  MAE  : {mae:.4f}  (avg absolute error in risk score)")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  R²   : {r2:.4f}")

    # Show score distribution of predictions
    low    = (y_pred <  0.33).sum()
    medium = ((y_pred >= 0.33) & (y_pred < 0.66)).sum()
    high   = (y_pred >= 0.66).sum()
    print(f"  Predicted zones  → Low: {low}  Medium: {medium}  High: {high}")


───────────────────────────────────────────────────────
  Training: Ridge Regression
───────────────────────────────────────────────────────
  MAE  : 0.0964  (avg absolute error in risk score)
  RMSE : 0.1343
  R²   : 0.8445
  Predicted zones  → Low: 27  Medium: 31  High: 79

───────────────────────────────────────────────────────
  Training: Random Forest
───────────────────────────────────────────────────────
  MAE  : 0.0830  (avg absolute error in risk score)
  RMSE : 0.1352
  R²   : 0.8422
  Predicted zones  → Low: 25  Medium: 30  High: 82

───────────────────────────────────────────────────────
  Training: XGBoost
───────────────────────────────────────────────────────
  MAE  : 0.0783  (avg absolute error in risk score)
  RMSE : 0.1335
  R²   : 0.8463
  Predicted zones  → Low: 29  Medium: 27  High: 81


In [7]:
# ── Inference helper ────────────────────────────────────────────────────────

def score_to_label(score: float) -> str:
    """Convert a continuous [0, 1] risk score to a display label."""
    if score < 0.33:
        return "Low Risk"
    elif score < 0.66:
        return "Medium Risk"
    else:
        return "High Risk"


# Demo: apply to test set using the best model so far
best_name = min(results, key=lambda n: results[n]["MAE"])
demo_preds = results[best_name]["y_pred"][:5]

print(f"Sample predictions from {best_name}:")
print(f"{'Score':>10}  {'Pct':>8}  {'Label'}")
print("─" * 38)
for s in demo_preds:
    print(f"{s:>10.4f}  {s*100:>7.1f}%  {score_to_label(s)}")

Sample predictions from XGBoost:
     Score       Pct  Label
──────────────────────────────────────
    0.1017     10.2%  Low Risk
    0.0324      3.2%  Low Risk
    0.8711     87.1%  High Risk
    0.7337     73.4%  High Risk
    0.1477     14.8%  Low Risk


## 5. Summary Table

In [8]:
summary = pd.DataFrame([
    {
        "Model": name,
        "MAE":   r["MAE"],
        "RMSE":  r["RMSE"],
        "R2":    r["R2"],
    }
    for name, r in results.items()
]).sort_values("MAE", ascending=True).reset_index(drop=True)

print("\n=== SYMPTOMS — All Models Summary (sorted by MAE, lower is better) ===")
print(summary.to_string(index=False))
print(f"\n→ Best model: {summary.iloc[0]['Model']} (MAE={summary.iloc[0]['MAE']})")


=== SYMPTOMS — All Models Summary (sorted by MAE, lower is better) ===
           Model    MAE   RMSE     R2
         XGBoost 0.0783 0.1335 0.8463
   Random Forest 0.0830 0.1352 0.8422
Ridge Regression 0.0964 0.1343 0.8445

→ Best model: XGBoost (MAE=0.0783)


## 6. Cross-Validation on Top-2 Models

In [9]:
# Combine train + test for CV (use full data)
X_all = pd.concat([X_train, X_test], ignore_index=True)
y_all = np.concatenate([y_train, y_test])

cv   = KFold(n_splits=5, shuffle=True, random_state=42)
top2 = summary["Model"].values[:2]

print("5-Fold Cross-Validation (neg MAE) — Top 2 Models:")
for name in top2:
    model  = results[name]["model"]
    scores = cross_val_score(model, X_all, y_all,
                             cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1)
    mae_scores = -scores
    print(f"  {name:<22} mean={mae_scores.mean():.4f}  "
          f"std={mae_scores.std():.4f}  "
          f"folds={np.round(mae_scores, 4)}")

5-Fold Cross-Validation (neg MAE) — Top 2 Models:
  XGBoost                mean=0.0607  std=0.0041  folds=[0.0607 0.0614 0.0624 0.0532 0.0658]
  Random Forest          mean=0.0632  std=0.0047  folds=[0.0646 0.0694 0.065  0.0552 0.0617]


## 7. Save All Trained Models

In [10]:
for name, r in results.items():
    safe_name = name.lower().replace(" ", "_")
    path = f"../models/symptoms/{safe_name}.pkl"
    joblib.dump(r["model"], path)
    print(f"  Saved: {path}")

# Save risk metadata for inference
risk_meta = {
    "output_type":      "regression",
    "high_signal_cols": HIGH_SIGNAL_COLS,
    "score_range":      [0.0, 1.0],
    "thresholds":       {"low": 0.33, "medium": 0.66},   # for label display
    "label_names":      ["Low Risk", "Medium Risk", "High Risk"],
}
joblib.dump(risk_meta, "../models/symptoms/risk_meta.pkl")

# Save summary for evaluation notebook
summary.to_csv("../models/symptoms/training_summary.csv", index=False)

print("\nAll models saved ✓")
print(f"Best model: {summary.iloc[0]['Model']} (MAE={summary.iloc[0]['MAE']})")

  Saved: ../models/symptoms/ridge_regression.pkl
  Saved: ../models/symptoms/random_forest.pkl
  Saved: ../models/symptoms/xgboost.pkl

All models saved ✓
Best model: XGBoost (MAE=0.0783)


## Summary

```
Task    : Continuous PCOS Risk Score → displayed as percentage (0–100 %)
Dataset : Symptoms — encoded + SMOTE balanced
Models  : Ridge Regression, Random Forest, XGBoost
Metrics : MAE, RMSE, R²
Zones   : Low < 33 % | 33 % ≤ Medium < 66 % | High ≥ 66 %

Saved artifacts:
  ../models/symptoms/ridge_regression.pkl
  ../models/symptoms/random_forest.pkl
  ../models/symptoms/xgboost.pkl
  ../models/symptoms/risk_meta.pkl
  ../models/symptoms/training_summary.csv
```